In [ ]:
!pip install langchain langchain-community langchain-text-splitters \
             langchain-huggingface langchain-groq \
             faiss-cpu sentence-transformers pypdf -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [ ]:
# PyPDFLoader = tool that reads PDF files (from disk OR internet URL)
from langchain_community.document_loaders import PyPDFLoader

# "https://arxiv.org/pdf/1706.03762" = URL of "Attention Is All You Need" paper
# No upload needed — loader fetches directly from internet
loader = PyPDFLoader("https://arxiv.org/pdf/1706.03762")

# .load() = actually read the PDF, return list of pages
# each page = one Document object (text + metadata)
pages = loader.load()

# f"..." = formatted string, lets us put variable inside {}
# len(pages) = count how many pages loaded
print(f"Total pages: {len(pages)}")

Total pages: 15


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
# RecursiveCharacterTextSplitter = smartest chunker in LangChain
# tries to split on paragraphs first, then sentences, then words
# "Recursive" = keeps trying smaller splits until chunk fits

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,    # each chunk = max 500 characters
    chunk_overlap=50   # last 50 chars of chunk 1 = first 50 chars of chunk 2
                       # prevents sentences cut in half losing meaning
)

# split_documents() = takes list of pages, returns list of chunks
chunks = splitter.split_documents(pages)

print(f"Total chunks: {len(chunks)}")
# 15 pages → should give ~100+ chunks

Total chunks: 93


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# HuggingFaceEmbeddings = downloads embedding model from HuggingFace
# model_name = "sentence-transformers/all-MiniLM-L6-v2"
# all-MiniLM-L6-v2 = 6-layer transformer, converts text → 384 numbers (vector)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# FAISS.from_documents() = takes all 93 chunks, converts each to vector, stores all
# embeddings = the model that does the converting
vectorstore = FAISS.from_documents(chunks, embeddings)

# vectorstore.index.ntotal = count vectors stored inside FAISS
print(f"Vectors stored: {vectorstore.index.ntotal}")
# should print 93

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vectors stored: 93


In [ ]:
import os
# os = operating system module, lets Python talk to system environment

from google.colab import userdata
# userdata = Colab's secret vault, where GROQ_API_KEY stored safely

from langchain_groq import ChatGroq
# ChatGroq = LangChain's connector to Groq's free LLM API

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
# os.environ = dictionary of environment variables
# ["GROQ_API_KEY"] = key name
# userdata.get("GROQ_API_KEY") = fetch secret from Colab vault

llm = ChatGroq(
    model_name="llama-3.1-8b-instant", # free fast model on Groq
    temperature=0  # 0 = factual mode, no creativity, same answer every time
)

print("LLM connected!")

LLM connected!


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
# ChatPromptTemplate = builds structured prompts for LLM
# MessagesPlaceholder = empty slot — chat_history fills here at runtime

from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain
# create_history_aware_retriever = Prompt 1 — reformulates question using history
# create_retrieval_chain = connects retriever + answer chain together

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
# create_stuff_documents_chain = stuffs retrieved chunks into prompt for LLM

from langchain_core.messages import HumanMessage, AIMessage
# HumanMessage = stores one user message in chat_history
# AIMessage = stores one AI reply in chat_history

# --- PROMPT 1: reformulate question using history ---
contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", "Given chat history and latest question, "
               "reformulate it as standalone question. "
               "Do NOT answer — just reformulate."),
    MessagesPlaceholder("chat_history"), # history injected here
    ("human", "{input}"),               # latest question goes here
])

# --- HISTORY-AWARE RETRIEVER ---
# combines Prompt 1 + LLM + retriever
# rewrites question → then searches FAISS
history_aware_retriever = create_history_aware_retriever(
    llm,                          # LLM does the reformulating
    vectorstore.as_retriever(search_kwargs={"k": 3}), # k=3 = fetch top 3 chunks
    contextualize_q_prompt        # Prompt 1 used here
)

# --- PROMPT 2: answer using chunks ---
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using context below. "
               "If unknown, say you don't know. "
               "Keep answer concise. "
               "Context: {context}"),  # {context} = retrieved chunks go here
    MessagesPlaceholder("chat_history"), # history here too for full context
    ("human", "{input}"),
])

# --- ANSWER CHAIN ---
question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

# --- FULL RAG CHAIN ---
rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

print("Memory RAG chain ready!")

Memory RAG chain ready!


In [ ]:
# chat_history = empty list at start — no messages yet
# this is the WhatsApp scroll — starts blank, fills as we talk
chat_history = []

# while True = loop forever until we type "exit"
while True:

    # input() = waits for you to type a question
    # "You: " = just a label showing in terminal
    question = input("You: ")

    # if you type "exit" — break = stop the loop
    if question.lower() == "exit":
        break

    # rag_chain.invoke() = run the full RAG pipeline
    # "input" = your question
    # "chat_history" = full WhatsApp scroll passed every time
    response = rag_chain.invoke({
        "input": question,
        "chat_history": chat_history  # history sent with every question
    })

    # response["answer"] = LLM's final answer
    answer = response["answer"]

    # print AI answer
    print(f"AI: {answer}\n")

    # append THIS question to history as HumanMessage
    chat_history.append(HumanMessage(content=question))

    # append AI reply to history as AIMessage
    chat_history.append(AIMessage(content=answer))
    # NOW chat_history has 2 more messages than before
    # next question will send this updated scroll

You: What is the attention mechanism?
AI: The attention mechanism is a method used in deep learning models, particularly in natural language processing and computer vision, to focus on specific parts of the input data that are relevant to the task at hand. It allows the model to weigh the importance of different elements in the input sequence, such as words in a sentence, and use this information to compute a representation of the sequence.

You: exit
